## 1. Wikipedia Retrivers

In [1]:
!uv add wikipedia -q

In [2]:
# Defining the Wikipedia Retriever 
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=2, 
    doc_content_chars_max=3000, 
    lang="en"
)

C:\Users\CSE\AppData\Local\Temp\ipykernel_11612\353107114.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [5]:
# Sometime error might occur -> re-run this cell 
query = "Prime Minister of Nepal"
docs = retriever.invoke(query)
# print(docs)

In [8]:
for i, doc in enumerate(docs): 
    print(f"\nDocument {i+1}")
    print("-"*50)
    print(doc.page_content[:200])
    print(f"The length of the document:{len(doc.page_content)}")



Document 1
--------------------------------------------------
The Prime Minister of Nepal (Nepali: नेपालको प्रधानमन्त्री, romanized: Nēpālakō pradhānamantrī) is the head of government of Federal Democratic Republic of Nepal. The prime minister leads the Council 
The length of the document:3000

Document 2
--------------------------------------------------
The Prime Minister of Nepal is the head of government of the Federal Democratic Republic of Nepal and the chairperson of the Council of Ministers. Although the President of Nepal is the constitutional
The length of the document:3000


## 2. Vector Store Retriever

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(
    model_name = "Sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    embedding_function = embedding, 
    persist_directory = "chroma_db", 
    collection_name = "movies"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\CSE\AppData\Local\Temp\ipykernel_11612\2450159708.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [10]:
# Defining vector store Retriever
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k":10}
)

In [25]:
query_1 = """
An 11-year-old boy discovers he is a wizard and begins studying at a magical 
boarding school. As he learns magic and forms lasting friendships, he uncovers 
the truth about his family's past and confronts a powerful dark wizard.
"""

query_2 = """Forrest Gump is a simple man with a low I.Q. but good intentions.
"He is running through childhood with his best and only friend Jenny. His 'mama' 
teaches him the ways of life and leaves him to choose his destiny. Forrest joins 
the army for service in Vietnam, finding new friends called Dan and Bubba, he wins 
medals, creates a famous shrimp fishing fleet, inspires people to jog, starts a 
ping-pong craze, creates the smiley, writes bumper stickers and songs, donates to 
people and meets the president several times. However, this is all irrelevant to Forrest 
who can only think of his childhood sweetheart Jenny Curran, who has messed up her life. 
Although in the end all he wants to prove is that anyone can love anyone."""

result = vector_retriever.invoke(query_2)
# result

In [12]:
for i, doc in enumerate(result, start = 1): 
    print(f"Result {i}: {doc.metadata["title"]}")

Result 1:  Forrest Gump
Result 2:  Finding Forrester
Result 3:  On Deadly Ground
Result 4:  The Emerald Forest
Result 5:  Dead Men Don't Wear Plaid
Result 6:  What About Bob?
Result 7:  Good Neighbor Sam
Result 8:  The Apartment
Result 9:  The Big Hit
Result 10:  Igby Goes Down


### 3. Maximum Marginal Relevance (MMR)

In [13]:
mmr_retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs={"k": 4, "fetch_k":20}
)

In [14]:
mmr_results = mmr_retriever.invoke(query_2)
# mmr_results

In [15]:
for i, doc in enumerate(mmr_results, start = 1): 
    print(f"Result {i}: {doc.metadata["title"]}")

Result 1:  Forrest Gump
Result 2:  Good Neighbor Sam
Result 3:  The Big Hit
Result 4:  Jabberwocky


### 4. BM25 Retriever

In [17]:
!uv add rank-bm25 -q

In [19]:
import json 
from langchain_classic.schema import Document

with open("../1_Datasets/movies_data_5000.json", "r", encoding="utf-8") as f: 
    data = json.load(f)

docs = [Document(**item) for item in data]
# docs[:4]

In [20]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 5

In [26]:
bm25_result = bm25_retriever.invoke(query_1)
# bm25_result

In [27]:
for i, doc in enumerate(bm25_result, start = 1): 
    print(f"Result {i}: {doc.metadata["title"]}")

Result 1:  Harry Potter and the Philosopher's Stone
Result 2:  Birdcage Inn
Result 3:  Nightwatch
Result 4:  Jade
Result 5:  The Princess and the Warrior


### 5. Ensemble Retriever

In [28]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retrievers = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], 
    weights=[0.5, 0.5]
)

In [29]:
ensemble_result = ensemble_retrievers.invoke(query_1)
# ensemble_result

In [30]:
for i, doc in enumerate(ensemble_result, start = 1): 
    print(f"Result {i}: {doc.metadata["title"]}")

Result 1:  Harry Potter and the Philosopher's Stone
Result 2:  Harry Potter and the Order of the Phoenix
Result 3:  Birdcage Inn
Result 4:  The Sword in the Stone
Result 5:  Nightwatch
Result 6:  Harry Potter and the Goblet of Fire
Result 7:  Jade
Result 8:  Bogus
Result 9:  The Princess and the Warrior
Result 10:  About a Boy
Result 11:  Harry Potter and the Prisoner of Azkaban
Result 12:  The Craft
Result 13:  Journey to the Beginning of Time
Result 14:  My Tutor


### BONUS: Reranker

In [31]:

from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [32]:
pairs = [(query_1, doc.page_content)for doc in ensemble_result]
scores = reranker.predict(pairs)
scores

array([2.63151735e-01, 2.80819903e-03, 4.16178082e-05, 2.41439492e-02,
       3.50066432e-04, 6.84592361e-03, 1.17571020e-04, 3.21599212e-03,
       6.72482784e-05, 1.08203734e-03, 8.86022206e-03, 6.99000317e-04,
       2.78392487e-04, 1.05594576e-04], dtype=float32)

In [33]:
ranked_docs = sorted(
    zip(ensemble_result, scores), 
    key = lambda x: x[1], 
    reverse = True
)

In [34]:
for i, doc in enumerate(ranked_docs, start = 1): 
    print(doc[0].metadata["title"])

 Harry Potter and the Philosopher's Stone
 The Sword in the Stone
 Harry Potter and the Prisoner of Azkaban
 Harry Potter and the Goblet of Fire
 Bogus
 Harry Potter and the Order of the Phoenix
 About a Boy
 The Craft
 Nightwatch
 Journey to the Beginning of Time
 Jade
 My Tutor
 The Princess and the Warrior
 Birdcage Inn
